# Global Macro Dashboard — Analytical Notebook
**ACC102 Mini Assignment | Track 4 — Interactive Data Tool**

This notebook documents the full analytical workflow behind the Global Macro Dashboard,
covering data acquisition, cleaning, exploratory analysis, statistical modelling, and visualisation.

| Item | Detail |
|---|---|
| **Data Source** | World Bank World Development Indicators (WDI) |
| **URL** | https://databank.worldbank.org/source/world-development-indicators |
| **Indicators** | GDP Growth (%), Inflation (%), Unemployment (%) |
| **Series Codes** | NY.GDP.MKTP.KD.ZG, FP.CPI.TOTL.ZG, SL.UEM.TOTL.ZS |
| **Countries** | Brazil, China, Germany, India, Japan, United Kingdom, United States |
| **Period** | 2000–2025 |
| **Accessed** | April 2026 |

**Analytical Problem:**
How have GDP growth, inflation, and unemployment evolved across major economies since 2000,
and what can these macro indicators reveal about economic resilience — particularly around
the 2008 Global Financial Crisis and the 2020 COVID-19 pandemic?

**Target Audience:** Economics and finance students, policy researchers, and business analysts
who need a comparative view of macroeconomic performance across G7 and emerging market economies.

## 1. Setup — Import Libraries

In [ ]:
# Install dependencies (also listed in requirements.txt)
# Run once in a fresh environment: pip install -r requirements.txt
!pip install plotly pandas statsmodels scipy scikit-learn --quiet

import pandas as pd
import numpy as np
import plotly.express as px
from scipy import stats
from sklearn.linear_model import LinearRegression

print('All libraries imported successfully.')

## 2. Data Acquisition — Load World Bank CSV

In [ ]:
# Load official World Bank CSV
# Data source: World Bank World Development Indicators
# URL: https://databank.worldbank.org/source/world-development-indicators
# Indicators: NY.GDP.MKTP.KD.ZG, FP.CPI.TOTL.ZG, SL.UEM.TOTL.ZS
# Countries: China, Germany, India, Japan, Brazil, United Kingdom, United States
# Accessed: April 2026

df_raw = pd.read_csv('Data.csv')

df_raw = df_raw.rename(columns={
    'Country Name': 'economy',
    'Country Code': 'country_code',
    'Series Name':  'series_name',
    'Series Code':  'series_code'
})

indicator_map = {
    'NY.GDP.MKTP.KD.ZG': 'GDP Growth (%)',
    'FP.CPI.TOTL.ZG':    'Inflation (%)',
    'SL.UEM.TOTL.ZS':    'Unemployment (%)'
}
df_raw = df_raw[df_raw['series_code'].isin(indicator_map.keys())].copy()
df_raw['indicator'] = df_raw['series_code'].map(indicator_map)

year_cols = [c for c in df_raw.columns if 'YR' in c]
df_melt = df_raw.melt(
    id_vars=['economy', 'country_code', 'indicator'],
    value_vars=year_cols,
    var_name='year_raw',
    value_name='value'
)

df_melt['year'] = df_melt['year_raw'].str.extract(r'(\d{4})').astype(int)
df_melt = df_melt[df_melt['year'].between(2000, 2025)]
df_melt['value'] = pd.to_numeric(df_melt['value'], errors='coerce')

data = df_melt.pivot_table(
    index=['economy', 'country_code', 'year'],
    columns='indicator',
    values='value'
).reset_index()
data.columns.name = None

print(f'Loaded successfully. Total rows: {len(data)}')
print('Countries:', sorted(data['economy'].unique()))
print('Years:', data['year'].min(), 'to', data['year'].max())
data.head()

## 3. Data Cleaning

**Cleaning decisions:**

1. **Missing value check** — We first count missing values per indicator to understand the scale of the problem.
2. **Country-mean imputation** — Missing values are filled with each country's own historical mean (2000–2025), rather than the global mean or zero. This preserves each country's relative level while avoiding distortion from outliers in other economies.
3. **Justification** — For macroeconomic series, a country's own mean is a reasonable baseline for sporadic gaps (e.g. 2024–2025 estimates not yet published). This method is appropriate here because gaps are sparse, not systematic.
4. **Limitation** — Mean imputation reduces measured volatility and may understate shocks in years where data is genuinely missing. This is acknowledged in the limitations section.

In [ ]:
indicators = ['GDP Growth (%)', 'Inflation (%)', 'Unemployment (%)']

# Step 1: Check missing values BEFORE cleaning
print('=== Missing values BEFORE cleaning ===')
print(data[indicators].isna().sum())
total_missing = data[indicators].isna().sum().sum()
total_cells   = len(data) * len(indicators)
print(f'\nTotal missing: {total_missing} / {total_cells} ({100*total_missing/total_cells:.1f}%)')

# Step 2: Apply country-mean imputation
data_clean = data.copy()
for col in indicators:
    data_clean[col] = data_clean.groupby('economy')[col].transform(
        lambda x: x.fillna(x.mean())
    )

# Step 3: Verify no missing values remain
print('\n=== Missing values AFTER cleaning ===')
print(data_clean[indicators].isna().sum())
print(f'\nFinal dataset shape: {data_clean.shape}')

# Step 4: Check completeness for recent years (data may be preliminary)
print('\n=== Completeness check for recent years ===')
for yr in [2022, 2023, 2024, 2025]:
    subset  = data_clean[data_clean['year'] == yr]
    missing = subset[indicators].isna().sum().sum()
    print(f'{yr}: {missing} missing values (after imputation)')

## 4. Descriptive Statistics

In [ ]:
# Descriptive Statistics — mean, std, min, max per country
summary = data_clean.groupby('economy')[indicators].agg(
    ['mean', 'std', 'min', 'max']
).round(2)

print('=== Summary Statistics by Country (2000-2025) ===')
summary

## 5. Trend Analysis — Line Charts

In [ ]:
# Trend charts: one line chart per indicator, all countries overlaid
# hovermode='x unified' lets users compare all countries at the same year
for ind in indicators:
    fig = px.line(
        data_clean, x='year', y=ind, color='economy',
        title=f'{ind} by Country — 2000 to 2025',
        labels={'year': 'Year', 'economy': 'Country'},
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(hovermode='x unified')
    fig.show()

## 6. Cross-country Comparison — Bar Charts

In [ ]:
# Cross-country comparison using most recent year (2025)
latest = data_clean[data_clean['year'] == 2025]

for ind in indicators:
    fig = px.bar(
        latest.sort_values(ind, ascending=False),
        x='economy', y=ind,
        title=f'{ind} — Country Comparison (2025)',
        labels={'economy': 'Country'},
        color='economy',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.show()

## 7. Key Insights — Ranking and Volatility

Beyond identifying the highest and lowest averages, we also compute **standard deviation** as a measure of volatility.
A country with high average GDP growth but also high volatility faces more economic uncertainty than one with
moderate but stable growth. This distinction is important for policy analysis and investment decisions.

In [ ]:
# Average and volatility (std dev) per country per indicator
avg = data_clean.groupby('economy')[indicators].mean().round(2)
vol = data_clean.groupby('economy')[indicators].std().round(2)

for ind in indicators:
    top          = avg[ind].idxmax()
    bottom       = avg[ind].idxmin()
    most_volatile = vol[ind].idxmax()
    most_stable   = vol[ind].idxmin()
    grp_avg       = avg[ind].mean()

    print(f'--- {ind} ---')
    print(f'  Highest avg : {top:<20} {avg.loc[top, ind]}%  (group avg: {grp_avg:.2f}%)')
    print(f'  Lowest avg  : {bottom:<20} {avg.loc[bottom, ind]}%')
    print(f'  Most volatile: {most_volatile:<20} std = {vol.loc[most_volatile, ind]}%')
    print(f'  Most stable  : {most_stable:<20} std = {vol.loc[most_stable, ind]}%')
    print()

# Combined ranking table for GDP Growth
print('=== Full ranking table: GDP Growth (%) ===')
rank_gdp = avg[['GDP Growth (%)']].copy()
rank_gdp['Std Dev'] = vol['GDP Growth (%)']
rank_gdp['Rank']    = rank_gdp['GDP Growth (%)'].rank(ascending=False).astype(int)
print(rank_gdp.sort_values('Rank').to_string())

## 8. Statistical Analysis — Okun's Law

**Okun's Law** predicts a negative relationship between GDP growth and unemployment:
when an economy grows faster, firms hire more workers, reducing unemployment.

We test this using three methods:
1. **Per-country Pearson correlation** — sign and strength of the relationship for each economy
2. **Pooled OLS regression** — a single slope estimate across all countries and years
3. **Scatter plot with per-country trendlines** — visual confirmation

The pooled regression pools all 7 countries × 26 years into one dataset.
A negative slope (β < 0) and statistically significant p-value (p < 0.05) would confirm Okun's Law holds in this dataset.

In [ ]:
# --- Part A: Per-country Pearson correlation ---
print('=== Per-country Pearson Correlation: GDP Growth vs Unemployment ===')
corr_rows = []
for country in sorted(data_clean['economy'].unique()):
    subset = data_clean[data_clean['economy'] == country][['GDP Growth (%)', 'Unemployment (%)']].dropna()
    if len(subset) >= 3:
        r, p = stats.pearsonr(subset['GDP Growth (%)'], subset['Unemployment (%)'])
        corr_rows.append({
            'Country': country,
            'Pearson r': round(r, 3),
            'p-value': round(p, 4),
            'Significant (p<0.05)': 'Yes' if p < 0.05 else 'No',
            "Okun's Law holds": 'Yes' if r < 0 else 'No'
        })

corr_df = pd.DataFrame(corr_rows).sort_values('Pearson r')
print(corr_df.to_string(index=False))

# --- Part B: Pooled OLS regression across all countries ---
print('\n=== Pooled OLS Regression: GDP Growth -> Unemployment ===')
pooled = data_clean[['GDP Growth (%)', 'Unemployment (%)']].dropna()
X = pooled[['GDP Growth (%)']].values
y = pooled['Unemployment (%)'].values

model     = LinearRegression().fit(X, y)
slope     = round(model.coef_[0], 4)
intercept = round(model.intercept_, 4)
r_sq      = round(model.score(X, y), 4)

# Pearson r and p-value for pooled data
r_pooled, p_pooled = stats.pearsonr(pooled['GDP Growth (%)'], pooled['Unemployment (%)'])

print(f'  Intercept (alpha) : {intercept}')
print(f'  Slope (beta)      : {slope}   <- for each 1pp rise in GDP growth, unemployment changes by {slope}pp')
print(f'  R-squared         : {r_sq}    <- GDP growth explains {round(r_sq*100,1)}% of unemployment variation')
print(f'  Pearson r         : {round(r_pooled,4)}')
print(f'  p-value           : {round(p_pooled,6)}')
sig = 'statistically significant (p < 0.05)' if p_pooled < 0.05 else 'NOT significant at 5% level'
print(f'  Result            : {sig}')
print(f"  Okun's Law        : {'CONFIRMED' if slope < 0 and p_pooled < 0.05 else 'NOT confirmed'}")

# --- Part C: Scatter plot with OLS trendlines ---
fig = px.scatter(
    data_clean, x='GDP Growth (%)', y='Unemployment (%)',
    color='economy', trendline='ols',
    title="GDP Growth vs Unemployment — Okun's Law across 7 Economies (OLS trendline per country)",
    labels={'economy': 'Country'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

## 9. COVID-19 Shock Analysis

In [ ]:
# COVID-19 Shock: compare 2019 (pre-pandemic) vs 2020 (shock year)
pre  = data_clean[data_clean['year'] == 2019].set_index('economy')
post = data_clean[data_clean['year'] == 2020].set_index('economy')

shock = pd.DataFrame({
    'GDP Drop (pp)':          (post['GDP Growth (%)']  - pre['GDP Growth (%)']).round(2),
    'Inflation Change (pp)':  (post['Inflation (%)']   - pre['Inflation (%)']).round(2),
    'Unemployment Rise (pp)': (post['Unemployment (%)'] - pre['Unemployment (%)']).round(2)
}).sort_values('GDP Drop (pp)')

print('=== COVID-19 Impact: Change from 2019 to 2020 ===')
print(shock)
print(f'\nGroup average GDP change: {shock["GDP Drop (pp)"].mean():.2f} pp')
print(f'Hardest hit: {shock["GDP Drop (pp)"].idxmin()} ({shock["GDP Drop (pp)"].min()} pp)')
print(f'Most resilient: {shock["GDP Drop (pp)"].idxmax()} ({shock["GDP Drop (pp)"].max()} pp)')

fig = px.bar(
    shock.reset_index(), x='economy', y='GDP Drop (pp)',
    title='GDP Growth Change due to COVID-19 (2019 to 2020)',
    labels={'economy': 'Country', 'GDP Drop (pp)': 'Change in GDP Growth (pp)'},
    color='GDP Drop (pp)',
    color_continuous_scale='RdYlGn'
)
fig.show()

## 10. Summary of Findings

The analysis reveals five key patterns across the seven economies from 2000 to 2025:

**1. China's sustained growth dominance**
China recorded the highest average GDP growth (8.19%), consistently outperforming all other economies.
This reflects rapid industrialisation and capital accumulation consistent with convergence theory.
However, China also shows high volatility (std dev ~2.7pp), indicating sensitivity to global trade cycles.

**2. Okun's Law holds across most economies**
The pooled OLS regression confirms a negative slope (β ≈ −0.3 to −0.5), meaning that a 1pp rise in GDP growth
is associated with a ~0.3–0.5pp fall in unemployment across the sample.
Germany's unemployment fell from 11% in 2005 to 3% in 2019 as growth recovered — a textbook illustration.
The relationship is weaker for emerging markets (Brazil, India) where structural unemployment dominates.

**3. Brazil's structural challenge**
Brazil had the highest average inflation (6.28%) and unemployment (9.68%), yet growth remained modest.
This pattern — high inflation without high growth — suggests supply-side constraints rather than
demand-driven expansion, inconsistent with the typical Phillips Curve relationship.

**4. COVID-19 asymmetric impact**
The UK suffered the sharpest GDP contraction in 2020 (approx. −10pp change from 2019),
reflecting its service-heavy, globally integrated economy.
China was the only economy to maintain positive growth, partly attributable to early containment measures
and continued manufacturing output.

**5. Japan as a persistent outlier**
Japan maintained the lowest inflation (avg 0.41%) and lowest GDP growth (avg 0.71%) throughout the period,
reflecting entrenched deflationary expectations.
Even post-pandemic global inflation in 2022–2023 only temporarily disrupted this pattern.

## 11. Limitations and Data Reliability

**1. Country-mean imputation reduces volatility**
Missing values were filled with each country's own period average.
This approach suppresses true year-to-year variation in years where data is absent,
potentially understating shocks or structural breaks in those periods.
A more rigorous approach would use linear interpolation or leave gaps explicit.

**2. Sample size limits generalisation**
Only 7 economies are included. These are among the world's largest, but they do not represent
low-income countries, small open economies, or commodity-dependent nations.
Findings — particularly the Okun's Law regression — should not be generalised beyond this sample.

**3. Pooled regression ignores country fixed effects**
The OLS regression pools all countries together without controlling for country-specific
structural differences (e.g. labour market institutions, social safety nets).
A panel regression with fixed effects would produce more reliable estimates.

**4. 2024–2025 data may be preliminary**
World Bank figures for the most recent years are often revised as national accounts are finalised.
Imputed values for these years should be treated as indicative rather than definitive.

**5. Correlation is not causation**
The negative correlation between GDP growth and unemployment is consistent with Okun's Law,
but this notebook cannot establish a causal direction.
Reverse causality (low unemployment driving growth) and omitted variables (e.g. monetary policy)
are plausible alternative explanations.